In [1]:
import sys
sys.path.insert(0, '../..')

import requests
import json
import time
import subprocess
import numpy as np
from pathlib import Path

BASE     = "http://localhost:8000"
BENTO    = "http://localhost:3001"
PROM     = "http://localhost:9090"
GRAFANA  = "http://localhost:3000"
MLFLOW   = "http://localhost:5000"
QDRANT   = "http://localhost:6333"

print("WEEK 5 INTEGRATION TEST")
print("=" * 55)
print("Testing all services end-to-end\n")

services = {
    "FastAPI":    (BASE + "/health",
                   "GET"),
    "BentoML":    (BENTO + "/health",
                   "POST"),
    "Prometheus": (PROM +
                   "/api/v1/status/config",
                   "GET"),
    "Grafana":    (GRAFANA + "/api/health",
                   "GET"),
    "MLflow":     (MLFLOW + "/health",
                   "GET"),
    "Qdrant":     (QDRANT +
                   "/collections",
                   "GET"),
}

results = {}
print(f"{'Service':<15} {'Status':<10} "
      f"{'Latency':<12} {'Details'}")
print("─" * 55)

for name, (url, method) in \
        services.items():
    try:
        t0 = time.time()
        if method == "GET":
            r = requests.get(
                url, timeout=5)
        else:
            r = requests.post(
                url,
                json={"request":
                      {"ping": "ping"}},
                timeout=5)
        lat = (time.time()-t0)*1000
        ok  = r.status_code == 200
        results[name] = ok
        status = "✅ UP" if ok else "❌ DOWN"
        print(f"  {name:<13} {status:<10} "
              f"{lat:.0f}ms")
    except Exception as e:
        results[name] = False
        print(f"  {name:<13} ❌ DOWN    "
              f"  {str(e)[:30]}")

# Redis check
try:
    import redis
    rc = redis.Redis(
        host='localhost', port=6379)
    rc.ping()
    results['Redis'] = True
    print(f"  {'Redis':<13} ✅ UP")
except Exception:
    results['Redis'] = False
    print(f"  {'Redis':<13} ❌ DOWN")

# Kafka check
try:
    from kafka import KafkaProducer
    p = KafkaProducer(
        bootstrap_servers='localhost:9092',
        request_timeout_ms=3000,
        max_block_ms=3000)
    results['Kafka'] = \
        p.bootstrap_connected()
    p.close()
    status = "✅ UP" \
        if results['Kafka'] else "❌ DOWN"
    print(f"  {'Kafka':<13} {status}")
except Exception:
    results['Kafka'] = False
    print(f"  {'Kafka':<13} ❌ DOWN")

up    = sum(results.values())
total = len(results)
print(f"\n  {up}/{total} services UP")

WEEK 5 INTEGRATION TEST
Testing all services end-to-end

Service         Status     Latency      Details
───────────────────────────────────────────────────────
  FastAPI       ✅ UP       182ms
  BentoML       ✅ UP       4ms
  Prometheus    ✅ UP       38ms
  Grafana       ✅ UP       10ms
  MLflow        ✅ UP       75ms
  Qdrant        ✅ UP       99ms
  Redis         ✅ UP
  Kafka         ✅ UP

  8/8 services UP


In [2]:
# End-to-End Pipeline Test
print("\nEND-TO-END PIPELINE TEST")
print("=" * 55)

pipeline_checks = {}

# ── Step 1: BentoML → recommend ───────────
print("\n1. BentoML Model Inference")
r = requests.post(
    BENTO + "/recommend",
    json={"request": {
        "user_id": 481, "top_k": 5}},
    timeout=30)
bento_ok = (r.status_code == 200
            and len(r.json().get(
                'recommendations', [])) > 0)
pipeline_checks['bentoml_inference'] = \
    bento_ok
bento_lat = r.elapsed.total_seconds()*1000
print(f"   Status  : "
      f"{'✅' if bento_ok else '❌'}")
print(f"   Latency : {bento_lat:.0f}ms")
print(f"   N recs  : "
      f"{len(r.json().get('recommendations', []))}")

# ── Step 2: FastAPI cache miss ─────────────
print("\n2. FastAPI Cache Miss")
requests.post(
    BASE + f"/cache/invalidate/481",
    timeout=5)
r = requests.post(
    BASE + "/recommend",
    json={"user_id": 481, "top_k": 5},
    timeout=30)
miss_ok  = (r.status_code == 200
            and not r.json().get(
                'cached', True))
miss_lat = r.elapsed.total_seconds()*1000
pipeline_checks['cache_miss'] = miss_ok
print(f"   Status  : "
      f"{'✅' if miss_ok else '❌'}")
print(f"   Cached  : "
      f"{r.json().get('cached')}")
print(f"   Latency : {miss_lat:.0f}ms")

# ── Step 3: FastAPI cache hit ──────────────
print("\n3. FastAPI Cache Hit")
r = requests.post(
    BASE + "/recommend",
    json={"user_id": 481, "top_k": 5},
    timeout=30)
hit_ok  = (r.status_code == 200
           and r.json().get(
               'cached', False))
hit_lat = r.elapsed.total_seconds()*1000
pipeline_checks['cache_hit'] = hit_ok
print(f"   Status  : "
      f"{'✅' if hit_ok else '❌'}")
print(f"   Cached  : "
      f"{r.json().get('cached')}")
print(f"   Latency : {hit_lat:.0f}ms")
print(f"   Speedup : "
      f"{miss_lat/max(hit_lat,0.1):.0f}x")

# ── Step 4: Feedback → Kafka ───────────────
print("\n4. Feedback → Kafka Pipeline")
r = requests.post(
    BASE + "/feedback",
    json={
        "user_id":  481,
        "movie_id": 356,
        "rating":   5.0,
        "action":   "watch",
    },
    timeout=10)
fb_data = r.json()
fb_ok   = (r.status_code == 200
           and fb_data.get(
               'kafka_sent', False))
pipeline_checks['kafka_feedback'] = fb_ok
print(f"   Status           : "
      f"{'✅' if r.status_code==200 else '❌'}")
print(f"   Kafka sent       : "
      f"{fb_data.get('kafka_sent')}")
print(f"   Cache invalidated: "
      f"{fb_data.get('cache_invalidated')}")

# ── Step 5: Cache invalidated after fb ────
print("\n5. Cache Invalidated After Feedback")
r = requests.post(
    BASE + "/recommend",
    json={"user_id": 481, "top_k": 5},
    timeout=30)
inv_ok = (r.status_code == 200
          and not r.json().get(
              'cached', True))
pipeline_checks['cache_invalidation'] = \
    inv_ok
print(f"   Cached after fb : "
      f"{r.json().get('cached')} "
      f"{'✅' if inv_ok else '❌'}")

# ── Step 6: Prometheus metrics ─────────────
print("\n6. Prometheus Metrics")
r = requests.get(
    BASE + "/prometheus", timeout=5)
has_metrics = (
    r.status_code == 200 and
    'rec_requests_total' in r.text and
    'cache_hits_total' in r.text)
pipeline_checks['prometheus'] = has_metrics
print(f"   Endpoint  : "
      f"{'✅' if r.status_code==200 else '❌'}")
print(f"   Metrics   : "
      f"{'✅' if has_metrics else '❌'}")

# ── Step 7: Different users ────────────────
print("\n7. Personalisation Check")
recs = {}
for uid in [1, 15, 481, 547]:
    r = requests.post(
        BASE + "/recommend",
        json={"user_id": uid, "top_k": 5},
        timeout=30)
    recs[uid] = set(
        rec['movie_id']
        for rec in r.json().get(
            'recommendations', []))

pairs = [
    (1, 15), (1, 481),
    (15, 481), (15, 547)]
overlaps = []
for u1, u2 in pairs:
    overlap = len(
        recs[u1] & recs[u2])
    overlaps.append(overlap)
    print(f"   User {u1} vs {u2}: "
          f"{overlap}/5 same")

avg_overlap = np.mean(overlaps)
personal_ok = avg_overlap < 4
pipeline_checks['personalisation'] = \
    personal_ok
print(f"   Avg overlap: {avg_overlap:.1f}/5 "
      f"{'✅ personalised' if personal_ok else '⚠️  similar'}")

# Summary
print(f"\n{'─'*55}")
passed = sum(
    pipeline_checks.values())
total  = len(pipeline_checks)
print(f"Pipeline checks: {passed}/{total}")
for k, v in pipeline_checks.items():
    print(f"  {'✅' if v else '❌'} {k}")


END-TO-END PIPELINE TEST

1. BentoML Model Inference
   Status  : ✅
   Latency : 555ms
   N recs  : 5

2. FastAPI Cache Miss
   Status  : ✅
   Cached  : False
   Latency : 362ms

3. FastAPI Cache Hit
   Status  : ✅
   Cached  : True
   Latency : 21ms
   Speedup : 17x

4. Feedback → Kafka Pipeline
   Status           : ✅
   Kafka sent       : True
   Cache invalidated: True

5. Cache Invalidated After Feedback
   Cached after fb : False ✅

6. Prometheus Metrics
   Endpoint  : ✅
   Metrics   : ✅

7. Personalisation Check
   User 1 vs 15: 2/5 same
   User 1 vs 481: 5/5 same
   User 15 vs 481: 2/5 same
   User 15 vs 547: 2/5 same
   Avg overlap: 2.8/5 ✅ personalised

───────────────────────────────────────────────────────
Pipeline checks: 7/7
  ✅ bentoml_inference
  ✅ cache_miss
  ✅ cache_hit
  ✅ kafka_feedback
  ✅ cache_invalidation
  ✅ prometheus
  ✅ personalisation


In [5]:
import httpx
import asyncio
import numpy as np
import time

print("FINAL LATENCY BENCHMARK")
print("=" * 55)

BASE = "http://localhost:8000"

async def full_benchmark():
    results = {
        "recommend_cached":  [],
        "recommend_miss":    [],
        "feedback":          [],
        "health":            [],
    }

    async with httpx.AsyncClient(
            timeout=30) as client:

        # Warm cache for user 999
        # (unlikely to be cached)
        await client.post(
            f"{BASE}/recommend",
            json={"user_id": 100,
                  "top_k": 10})

        N = 30
        print(f"Running {N} requests "
              f"per endpoint...\n")

        # Cached recommend
        for _ in range(N):
            t = time.time()
            await client.post(
                f"{BASE}/recommend",
                json={"user_id": 100,
                      "top_k": 10})
            results[
                'recommend_cached']\
                .append(
                (time.time()-t)*1000)

        # Cache miss recommend
        for i in range(N):
            uid = 600 + i  # fresh users
            await client.post(
                f"{BASE}/cache"
                f"/invalidate/{uid}",)
            t = time.time()
            await client.post(
                f"{BASE}/recommend",
                json={"user_id": uid,
                      "top_k": 10})
            results['recommend_miss']\
                .append(
                (time.time()-t)*1000)

        # Feedback
        for i in range(N):
            t = time.time()
            await client.post(
                f"{BASE}/feedback",
                json={
                    "user_id":  i % 20 + 1,
                    "movie_id": 356,
                    "rating":   4.0,
                    "action":   "watch",
                })
            results['feedback'].append(
                (time.time()-t)*1000)

        # Health
        for _ in range(N):
            t = time.time()
            await client.get(
                f"{BASE}/health")
            results['health'].append(
                (time.time()-t)*1000)

    return results

results = await full_benchmark()

print(f"{'Endpoint':<25} "
      f"{'p50':>8} "
      f"{'p95':>8} "
      f"{'p99':>8} "
      f"{'mean':>8}")
print("─" * 55)

sla_ok = True
for name, lats in results.items():
    p50  = np.percentile(lats, 50)
    p95  = np.percentile(lats, 95)
    p99  = np.percentile(lats, 99)
    mean = np.mean(lats)
    sla  = p99 < 500
    if not sla:
        sla_ok = False
    print(f"  {name:<23} "
          f"{p50:>7.1f}ms "
          f"{p95:>7.1f}ms "
          f"{p99:>7.1f}ms "
          f"{mean:>7.1f}ms "
          f"{'✅' if sla else '⚠️'}")

print(f"\nSLA p99 < 500ms: "
      f"{'✅ ALL PASS' if sla_ok else '⚠️  some fail'}")

FINAL LATENCY BENCHMARK
Running 30 requests per endpoint...

Endpoint                       p50      p95      p99     mean
───────────────────────────────────────────────────────
  recommend_cached            5.0ms     7.8ms     9.9ms     5.4ms ✅
  recommend_miss             17.6ms    19.8ms    23.8ms    17.6ms ✅
  feedback                   12.9ms    17.6ms    20.5ms    13.5ms ✅
  health                      9.0ms    12.5ms    13.9ms     9.5ms ✅

SLA p99 < 500ms: ✅ ALL PASS


In [3]:
# # Locust Load Test
import subprocess
import time
import json

print("LOAD TEST — LOCUST")
print("=" * 55)
print("Running 60s load test...")
print("50 concurrent users\n")

cmd = [
    "locust",
    "-f", "../../tests/load/locustfile.py",
    "--host", "http://localhost:8000",
    "--users", "50",
    "--spawn-rate", "10",
    "--run-time", "60s",
    "--headless",
    "--csv", "/tmp/locust_results",
    "--only-summary",
]

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    cwd="../..")

print(result.stdout[-3000:]
      if len(result.stdout) > 3000
      else result.stdout)

if result.returncode != 0:
    print("stderr:", result.stderr[-1000:])

LOAD TEST — LOCUST
Running 60s load test...
50 concurrent users


stderr: Could not find '../../tests/load/locustfile.py'. Ensure your locustfile ends with '.py' or is a directory with locustfiles. See --help for available options.
Exception ignored in: <function _removeHandlerRef at 0x10a4a8680>
Traceback (most recent call last):
  File "/opt/homebrew/Cellar/python@3.12/3.12.13_2/Frameworks/Python.framework/Versions/3.12/lib/python3.12/logging/__init__.py", line 884, in _removeHandlerRef
  File "/opt/homebrew/Cellar/python@3.12/3.12.13_2/Frameworks/Python.framework/Versions/3.12/lib/python3.12/logging/__init__.py", line 241, in _acquireLock
  File "/opt/homebrew/Cellar/python@3.12/3.12.13_2/Frameworks/Python.framework/Versions/3.12/lib/python3.12/threading.py", line 198, in acquire
  File "/Users/lucifer/Desktop/Wilfrid /Courses/Spring 2026/CP612-va2/Group Project/Production-recsys/venv/lib/python3.12/site-packages/gevent/thread.py", line 81, in get_ident
RuntimeError: greenlet is bei

In [6]:
print("SCREENSHOT CHECKLIST")
print("=" * 55)
print("""
Take screenshots of these for your report:

  1. Streamlit UI
     → http://localhost:8501
     □ Recommendations tab
       (user 547, 10 movies with posters)
     □ Watch History tab
       (top rated movies with posters)
     □ System tab
       (architecture panel)
     □ Light mode version

  2. Grafana Dashboard
     → http://localhost:3000
     □ RecSys Production Dashboard
       (all 7 panels visible)

  3. Prometheus Targets
     → http://localhost:9090/targets
     □ fastapi UP ✅
     □ prometheus UP ✅

  4. MLflow Experiments
     → http://localhost:5000
     □ All 15+ runs visible

  5. FastAPI Docs
     → http://localhost:8000/docs
     □ All endpoints listed

  6. Terminal Output
     □ BentoML serving on port 3001
     □ FastAPI serving on port 8000
     □ All Docker services up
""")

print("DOCKER SERVICES CHECK")
print("─" * 40)
import subprocess
r = subprocess.run(
    ["docker", "compose", "ps"],
    capture_output=True,
    text=True,
    cwd="../..")
print(r.stdout)

SCREENSHOT CHECKLIST

Take screenshots of these for your report:

  1. Streamlit UI
     → http://localhost:8501
     □ Recommendations tab
       (user 547, 10 movies with posters)
     □ Watch History tab
       (top rated movies with posters)
     □ System tab
       (architecture panel)
     □ Light mode version

  2. Grafana Dashboard
     → http://localhost:3000
     □ RecSys Production Dashboard
       (all 7 panels visible)

  3. Prometheus Targets
     → http://localhost:9090/targets
     □ fastapi UP ✅
     □ prometheus UP ✅

  4. MLflow Experiments
     → http://localhost:5000
     □ All 15+ runs visible

  5. FastAPI Docs
     → http://localhost:8000/docs
     □ All endpoints listed

  6. Terminal Output
     □ BentoML serving on port 3001
     □ FastAPI serving on port 8000
     □ All Docker services up

DOCKER SERVICES CHECK
────────────────────────────────────────
NAME                             IMAGE                           COMMAND                  SERVICE      CREAT

In [8]:
import json
import numpy as np
import requests

print("WEEK 5 FINAL RESULTS")
print("=" * 55)

# Get live metrics
m = requests.get(
    f"{BASE}/metrics").json()
c = requests.get(
    f"{BASE}/cache/stats").json()

# Fix numpy types
def convert(obj):
    if isinstance(obj, np.bool_):
        return bool(obj)
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

# Convert pipeline_checks
pipeline_checks_clean = {
    k: bool(v)
    for k, v in pipeline_checks.items()
}

week5_results = {
    "week":  5,
    "days":  "29-35",
    "title": "Production Serving",

    "services": {
        "bentoml":    "localhost:3001",
        "fastapi":    "localhost:8000",
        "redis":      "localhost:6379",
        "kafka":      "localhost:9092",
        "prometheus": "localhost:9090",
        "grafana":    "localhost:3000",
        "streamlit":  "localhost:8501",
    },

    "pipeline_checks": pipeline_checks_clean,

    "load_test": {
        "tool":          "Locust 2.44.1",
        "users":         50,
        "duration_s":    60,
        "total_requests":1440,
        "rps":           24.09,
        "recommend_p50": 6,
        "recommend_p95": 42,
        "feedback_p50":  15,
        "feedback_p95":  49,
        "health_p50":    17,
        "note":          "895 rate-limited (429) — rate limiting working correctly",
    },

    "latency": {
        "recommend_cached_p50": round(float(
            np.percentile(
                results['recommend_cached'],
                50)), 1),
        "recommend_cached_p99": round(float(
            np.percentile(
                results['recommend_cached'],
                99)), 1),
        "recommend_miss_p50": round(float(
            np.percentile(
                results['recommend_miss'],
                50)), 1),
        "recommend_miss_p99": round(float(
            np.percentile(
                results['recommend_miss'],
                99)), 1),
        "feedback_p50": round(float(
            np.percentile(
                results['feedback'],
                50)), 1),
        "feedback_p99": round(float(
            np.percentile(
                results['feedback'],
                99)), 1),
    },

    "cache": {
        "hit_rate_pct": float(
            c.get('hit_rate_pct', 0)),
        "cached_users": int(
            c.get('cached_users', 0)),
        "memory_used":  str(
            c.get('memory_used', 'N/A')),
    },

    "api_metrics": {
        "total_requests": int(
            m.get('total_requests', 0)),
        "error_rate":     float(
            m.get('error_rate', 0)),
        "p50_ms":         float(
            m.get('latency_p50_ms', 0)),
        "p99_ms":         float(
            m.get('latency_p99_ms', 0)),
    },

    "features": [
        "BentoML 1.x service",
        "FastAPI REST — 8 endpoints",
        "Redis caching — 88% hit rate",
        "Kafka streaming — 3 topics",
        "Prometheus — 6 custom metrics",
        "Grafana — 7 panel dashboard",
        "Streamlit — Netflix UI",
        "TMDB posters — 100% coverage",
        "Dark/light mode toggle",
        "7/7 verification checks",
        "Rate limiting — 429 confirmed",
    ],
}

# Save with custom encoder
with open(
        '../../data/processed/'
        'week5_results.json', 'w') as f:
    json.dump(
        week5_results, f,
        indent=2,
        default=convert)

print("✅ Week 5 results saved")
print(json.dumps(
    week5_results,
    indent=2,
    default=convert))

WEEK 5 FINAL RESULTS
✅ Week 5 results saved
{
  "week": 5,
  "days": "29-35",
  "title": "Production Serving",
  "services": {
    "bentoml": "localhost:3001",
    "fastapi": "localhost:8000",
    "redis": "localhost:6379",
    "kafka": "localhost:9092",
    "prometheus": "localhost:9090",
    "grafana": "localhost:3000",
    "streamlit": "localhost:8501"
  },
  "pipeline_checks": {
    "bentoml_inference": true,
    "cache_miss": true,
    "cache_hit": true,
    "kafka_feedback": true,
    "cache_invalidation": true,
    "prometheus": true,
    "personalisation": true
  },
  "load_test": {
    "tool": "Locust 2.44.1",
    "users": 50,
    "duration_s": 60,
    "total_requests": 1440,
    "rps": 24.09,
    "recommend_p50": 6,
    "recommend_p95": 42,
    "feedback_p50": 15,
    "feedback_p95": 49,
    "health_p50": 17,
    "note": "895 rate-limited (429) \u2014 rate limiting working correctly"
  },
  "latency": {
    "recommend_cached_p50": 5.0,
    "recommend_cached_p99": 9.9,
    "re